# 02 -- Modelo neural (MLP sobre as features do fe_v4)

Continuação do sandbox de `01_eda.ipynb`. Decisão de arquitetura: um MLP
(PyTorch) treinado sobre as MESMAS features tabulares do `fe_v4` (co-visitação,
afinidade de categoria/categoria-pai, popularidade relativa, etc.), em vez de
um modelo de embeddings colaborativo (NCF puro). Razão: reaproveita o
`evaluate_model()` e as features já validadas, permitindo comparação direta
com os 5 modelos tabulares -- e evita redescobrir o que o projeto original já
mostrou (decisões #19/#23 do projeto anterior): embeddings puros tendem a
perder para os modelos de árvore quando há features tabulares fortes.

**Organização no MLflow** -- categoria irmã de `tree-baseline-models`, sem
grupo de feature-engineering (decisão explícita: reaproveitar fe_v4 como
está):

```
neural-models  (categoria)
 ├─ mlp-fe_v4              (run única com hiperparâmetros default)
 └─ hyperparameter-tuning  (sub-grupo)
     └─ tuning-mlp-fe_v4
         ├─ trial-000 ... trial-N   (train/val/test completos, tag is_best)
```

Mesmo leaderboard mechanism de `01_eda.ipynb` (métricas "best_*" logadas
direto na run da categoria e do sub-grupo), só que apontando pra esta
categoria.

## 1. Carregar fe_v4 (persistido por `01_eda.ipynb`)

`train_feat_v4`/`val_feat_v4`/`test_feat_v4`/`train_eval_feat_v4` foram
gravados em `data/processed/fe_v4/` por uma célula nova em `01_eda.ipynb` --
evita refazer aqui as ~80 células de feature engineering só pra ter acesso as
mesmas tabelas.

In [1]:
import json
import random
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.model_selection import ParameterSampler
from sklearn.preprocessing import StandardScaler

import mlflow

SEED = 42


def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)


set_seed()
# nada de cuda aqui de propósito, o dataset e a rede são pequenos o suficiente pra rodar tranquilo em cpu
print("torch:", torch.__version__, "| device: cpu (CPU-only, conforme convenção do projeto)")

torch: 2.12.1+cpu | device: cpu (CPU-only, conforme convencao do projeto)


In [ ]:
MLFLOW_DIR = Path("mlflow-data").resolve()
MLFLOW_DIR.mkdir(parents=True, exist_ok=True)

tracking_db = MLFLOW_DIR / "mlflow.db"
mlflow.set_tracking_uri(f"sqlite:///{tracking_db.as_posix()}")

EXPERIMENT_NAME = "ecommerce-recsys"
if mlflow.get_experiment_by_name(EXPERIMENT_NAME) is None:
    artifact_location = (MLFLOW_DIR / "artifacts").resolve().as_uri()
    mlflow.create_experiment(EXPERIMENT_NAME, artifact_location=artifact_location)
mlflow.set_experiment(EXPERIMENT_NAME)

print("Tracking URI:", mlflow.get_tracking_uri())

In [3]:
FE_V4_DIR = Path("../data/processed/fe_v4").resolve()

train_feat_v4 = pd.read_parquet(FE_V4_DIR / "train.parquet")
val_feat_v4 = pd.read_parquet(FE_V4_DIR / "val.parquet")
test_feat_v4 = pd.read_parquet(FE_V4_DIR / "test.parquet")
train_eval_feat_v4 = pd.read_parquet(FE_V4_DIR / "train_eval.parquet")
FEATURE_COLUMNS_V4 = json.loads((FE_V4_DIR / "feature_columns.json").read_text())
# se esse read_parquet falhar, é porque o 01_eda.ipynb ainda não rodou até a seção 15b
all_items = pd.read_parquet(FE_V4_DIR / "all_items.parquet")["itemid"].to_numpy()

print(f"train: {len(train_feat_v4):,} linhas | val: {len(val_feat_v4):,} | test: {len(test_feat_v4):,}")
print("Features:", FEATURE_COLUMNS_V4)

train: 1,804,100 linhas | val: 9,163,000 | test: 9,163,000
Features: ['user_activity_count', 'item_popularity_count', 'item_recency_days', 'item_covisitation_cosine', 'category_affinity', 'parent_category_affinity', 'item_relative_popularity']


## 2. Avaliação e graficos (copiados de `01_eda.ipynb`, sem alterações)

Mesma `evaluate_model()` usada pelos 5 modelos tabulares -- funciona sem
mudança nenhuma desde que o modelo exponha `.predict_proba(X)` no formato
sklearn. É exatamente pra isso que o wrapper `NeuralMLPClassifier` existe
(seção 4).

In [4]:
def evaluate_model(model, eval_df, feature_columns, all_items, k=10):
    eval_df = eval_df.sort_values("visitorid")
    scores = model.predict_proba(eval_df[feature_columns])[:, 1]
    n_users = eval_df["visitorid"].nunique()
    assert len(eval_df) % n_users == 0, "grupo com tamanho irregular por usuário"
    group_size = len(eval_df) // n_users
    scores_matrix = scores.reshape(n_users, group_size)
    labels_matrix = eval_df["label"].to_numpy().reshape(n_users, group_size)
    items_matrix = eval_df["itemid"].to_numpy().reshape(n_users, group_size)
    order = np.argsort(-scores_matrix, axis=1)
    ranked_labels = np.take_along_axis(labels_matrix, order, axis=1)[:, :k]
    ranked_items = np.take_along_axis(items_matrix, order, axis=1)[:, :k]
    has_hit = ranked_labels.any(axis=1)
    discounts = 1.0 / np.log2(np.arange(2, k + 2))
    ndcg = (ranked_labels * discounts).sum(axis=1)
    first_hit_pos = np.argmax(ranked_labels, axis=1)
    mrr = np.where(has_hit, 1.0 / (first_hit_pos + 1), 0.0)
    coverage = len(np.unique(ranked_items)) / len(all_items)
    return {"ndcg": ndcg.mean(), "hit_rate": has_hit.mean(), "mrr": mrr.mean(), "coverage": coverage}


def plot_metrics_across_splits(train_metrics, val_metrics, test_metrics, title):
    metric_names = list(train_metrics.keys())
    fig, axes = plt.subplots(1, len(metric_names), figsize=(4 * len(metric_names), 3.5))
    for ax, name in zip(axes, metric_names):
        values = [train_metrics[name], val_metrics[name], test_metrics[name]]
        ax.bar(["train", "val", "test"], values, color=["#4C72B0", "#DD8452", "#55A868"])
        ax.set_title(name)
    fig.suptitle(title)
    plt.tight_layout()
    return fig

## 3. Padronização das features

Diferente das árvores/logreg (que usam as features em escala bruta), um MLP
treina por gradiente descendente e a escala de cada feature afeta a
convergência diretamente. O `StandardScaler` fica **dentro** do wrapper do
modelo (seção 4) -- ajustado sempre só nos dados passados a `.fit()` (nunca
em val/test), pra manter o mesmo contrato `.fit(X, y)` / `.predict_proba(X)`
que os modelos de árvore usam, sem precisar mudar `evaluate_model()` ou
`run_hyperparameter_search()`.

## 4. Modelo (`MLP`) e wrapper sklearn-like (`NeuralMLPClassifier`)

O wrapper existe pra encaixar o PyTorch no mesmo contrato usado pelos modelos
de árvore (`fit(X, y)` / `predict_proba(X)`), reaproveitando 100% do
`evaluate_model()` e do `run_hyperparameter_search()` já escritos.

Early stopping usa uma fatia interna de validação (10% do próprio `X`
passado a `fit()`), **separada** de `val_feat_v4` -- se usássemos
`val_feat_v4` tanto pra early stopping quanto pra escolher o melhor trial
(por `val_ndcg`), estaríamos otimizando duas vezes no mesmo conjunto.

In [ ]:
class MLP(nn.Module):
    def __init__(self, n_features, hidden_dims=(64, 32), dropout=0.2):
        super().__init__()
        layers = []
        in_dim = n_features
        for h in hidden_dims:
            layers += [nn.Linear(in_dim, h), nn.ReLU(), nn.Dropout(dropout)]
            in_dim = h
        layers.append(nn.Linear(in_dim, 1))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x).squeeze(-1)


class NeuralMLPClassifier:
    """Wrapper sklearn-like (fit/predict_proba) em torno do MLP, pra encaixar
    sem alterações em evaluate_model() e run_hyperparameter_search().

    `weighted_loss` liga/desliga o pos_weight na BCE -- existe como parametro
    (em vez de sempre ligado) para permitir comparar "baseline" (config
    original: BCE simples, patience=5) contra "tuned" (pos_weight, patience=10)
    nos MESMOS hiperparâmetros sorteados, isolando o efeito dessas duas
    mudanças do resto da busca."""

    def __init__(self, hidden_dims=(64, 32), dropout=0.2, lr=1e-3, weight_decay=0.0,
                 batch_size=512, max_epochs=100, patience=10, weighted_loss=True, seed=SEED):
        self.hidden_dims = tuple(hidden_dims)
        self.dropout = dropout
        self.lr = lr
        self.weight_decay = weight_decay
        self.batch_size = batch_size
        self.max_epochs = max_epochs
        self.patience = patience
        self.weighted_loss = weighted_loss
        self.seed = seed

    def get_params(self, deep=True):
        return {
            "hidden_dims": self.hidden_dims, "dropout": self.dropout, "lr": self.lr,
            "weight_decay": self.weight_decay, "batch_size": self.batch_size,
            "max_epochs": self.max_epochs, "patience": self.patience,
            "weighted_loss": self.weighted_loss,
        }

    def fit(self, X, y):
        set_seed(self.seed)
        X = np.asarray(X, dtype="float32")
        y = np.asarray(y, dtype="float32")

        self.scaler_ = StandardScaler()
        self.scaler_.fit(X)

        n = len(X)
        rng = np.random.RandomState(self.seed)
        idx = rng.permutation(n)
        n_stop = max(1, int(n * 0.1))
        stop_idx, fit_idx = idx[:n_stop], idx[n_stop:]

        X_fit = self.scaler_.transform(X[fit_idx])
        X_stop = self.scaler_.transform(X[stop_idx])
        y_fit, y_stop = y[fit_idx], y[stop_idx]

        if self.weighted_loss:
            # Negative sampling é 1:4 (~20% positivos) -- sem isso, a BCE trata
            # um acerto e um erro de negativo como igualmente importantes, mas
            # o que a gente avalia é ranking (colocar o positivo acima dos
            # negativos). pos_weight = n_neg/n_pos reponde a loss em favor de
            # acertar os positivos, calculado a cada fit().
            n_pos = y_fit.sum()
            n_neg = len(y_fit) - n_pos
            self.pos_weight_ = float(n_neg / max(n_pos, 1.0))
            loss_fn = nn.BCEWithLogitsLoss(pos_weight=torch.tensor(self.pos_weight_, dtype=torch.float32))
        else:
            self.pos_weight_ = 1.0
            loss_fn = nn.BCEWithLogitsLoss()

        n_features = X_fit.shape[1]
        self.model_ = MLP(n_features, self.hidden_dims, self.dropout)
        optimizer = torch.optim.Adam(self.model_.parameters(), lr=self.lr, weight_decay=self.weight_decay)

        dataset = torch.utils.data.TensorDataset(torch.from_numpy(X_fit), torch.from_numpy(y_fit))
        generator = torch.Generator().manual_seed(self.seed)
        loader = torch.utils.data.DataLoader(
            dataset, batch_size=self.batch_size, shuffle=True, generator=generator
        )

        X_stop_t = torch.from_numpy(X_stop)
        y_stop_t = torch.from_numpy(y_stop)

        self.loss_history_ = {"train": [], "early_stop": []}
        best_loss = float("inf")
        best_state = None
        epochs_without_improvement = 0

        for _ in range(self.max_epochs):
            self.model_.train()
            running_loss = 0.0
            for xb, yb in loader:
                optimizer.zero_grad()
                loss = loss_fn(self.model_(xb), yb)
                loss.backward()
                optimizer.step()
                running_loss += loss.item() * len(xb)
            train_loss = running_loss / len(dataset)

            self.model_.eval()
            with torch.no_grad():
                stop_loss = loss_fn(self.model_(X_stop_t), y_stop_t).item()

            self.loss_history_["train"].append(train_loss)
            self.loss_history_["early_stop"].append(stop_loss)

            if stop_loss < best_loss - 1e-5:
                best_loss = stop_loss
                best_state = {k: v.clone() for k, v in self.model_.state_dict().items()}
                epochs_without_improvement = 0
            else:
                epochs_without_improvement += 1
                if epochs_without_improvement >= self.patience:
                    break

        if best_state is not None:
            self.model_.load_state_dict(best_state)
        self.n_epochs_trained_ = len(self.loss_history_["train"])
        return self

    def predict_proba(self, X):
        X = np.asarray(X, dtype="float32")
        X_scaled = self.scaler_.transform(X)
        self.model_.eval()
        with torch.no_grad():
            probs = torch.sigmoid(self.model_(torch.from_numpy(X_scaled))).numpy()
        return np.column_stack([1 - probs, probs])


print("MLP e NeuralMLPClassifier definidos (weighted_loss e patience configuráveis por trial).")

## 5. Teste rápido (smoke test, sem MLflow)

Antes de logar qualquer coisa, confirma que o wrapper treina e que
`evaluate_model()` aceita o modelo sem erros -- mesmo espírito do que foi
feito com negative sampling em `01_eda.ipynb` antes de aceitar os resultados.

In [6]:
_smoke_model = NeuralMLPClassifier(hidden_dims=(32,), max_epochs=3, patience=3)
_smoke_model.fit(train_feat_v4[FEATURE_COLUMNS_V4], train_feat_v4["label"])
_smoke_metrics = evaluate_model(_smoke_model, val_feat_v4, FEATURE_COLUMNS_V4, all_items, k=10)
print("Smoke test OK --", _smoke_metrics)
del _smoke_model, _smoke_metrics

Smoke test OK -- {'ndcg': np.float64(0.7534061485722688), 'hit_rate': np.float64(0.859871221215759), 'mrr': np.float64(0.7189647287486425), 'coverage': 0.8889370339118338}


## 6. Categoria `neural-models` no MLflow

In [7]:
NEURAL_CATEGORY_NAME = "neural_models"
neural_category_run_id_path = MLFLOW_DIR / "neural_category_run_id.txt"

if neural_category_run_id_path.exists():
    existing_run_id = neural_category_run_id_path.read_text().strip()
    neural_category_run = mlflow.start_run(run_id=existing_run_id)
    print(f"Retomando categoria existente: neural-models ({neural_category_run.info.run_id})")
else:
    neural_category_run = mlflow.start_run(run_name="neural-models")
    mlflow.set_tags({"category": NEURAL_CATEGORY_NAME})
    neural_category_run_id_path.write_text(neural_category_run.info.run_id)
    print(f"Categoria criada: neural-models ({neural_category_run.info.run_id})")

NEURAL_CATEGORY_RUN_ID = neural_category_run.info.run_id


def run_exists(run_name):
    exp = mlflow.get_experiment_by_name(EXPERIMENT_NAME)
    if exp is None:
        return False
    existing = mlflow.search_runs(
        experiment_ids=[exp.experiment_id],
        filter_string=f"tags.mlflow.runName = '{run_name}'",
    )
    return len(existing) > 0

Retomando categoria existente: neural-models (9e9eea809c214960ad1988d408c13bff)


## 7. Run baseline -- `mlp-fe_v4` (hiperparâmetros default)

Uma única run direto sob `neural-models`, com hiperparâmetros razoáveis
default -- antes de qualquer busca, pra ter um primeiro número pra comparar.

In [ ]:
if run_exists("mlp-fe_v4"):
    print("mlp-fe_v4 já logado, pulando.")
else:
    baseline_mlp = NeuralMLPClassifier(
        hidden_dims=(64, 32), dropout=0.2, lr=1e-3, weight_decay=0.0,
        batch_size=512, max_epochs=100, patience=10, seed=42,
    )
    baseline_mlp.fit(train_feat_v4[FEATURE_COLUMNS_V4], train_feat_v4["label"])

    train_metrics = evaluate_model(baseline_mlp, train_eval_feat_v4, FEATURE_COLUMNS_V4, all_items, k=10)
    val_metrics = evaluate_model(baseline_mlp, val_feat_v4, FEATURE_COLUMNS_V4, all_items, k=10)
    test_metrics = evaluate_model(baseline_mlp, test_feat_v4, FEATURE_COLUMNS_V4, all_items, k=10)

    with mlflow.start_run(run_name="mlp-fe_v4", nested=True):
        mlflow.set_tags({"model_family": "neural", "feature_set": "fe_v4", "category": NEURAL_CATEGORY_NAME})
        mlflow.log_params(baseline_mlp.get_params())
        mlflow.log_metric("n_epochs_trained", baseline_mlp.n_epochs_trained_)
        mlflow.log_metric("pos_weight", baseline_mlp.pos_weight_)
        mlflow.log_metrics({f"train_{k}": v for k, v in train_metrics.items()})
        mlflow.log_metrics({f"val_{k}": v for k, v in val_metrics.items()})
        mlflow.log_metrics({f"test_{k}": v for k, v in test_metrics.items()})

        fig_loss, ax = plt.subplots(figsize=(6, 4))
        ax.plot(baseline_mlp.loss_history_["train"], label="train")
        ax.plot(baseline_mlp.loss_history_["early_stop"], label="early-stop (interno)")
        ax.set_xlabel("epoch")
        ax.set_ylabel("BCE loss (com pos_weight)")
        ax.set_title("Curva de perda -- mlp-fe_v4")
        ax.legend()
        plt.tight_layout()
        mlflow.log_figure(fig_loss, "loss_curve.png")
        plt.close(fig_loss)

        fig_splits = plot_metrics_across_splits(train_metrics, val_metrics, test_metrics, "mlp-fe_v4")
        mlflow.log_figure(fig_splits, "metrics_by_split.png")
        plt.close(fig_splits)

    print(f"mlp-fe_v4 -- test_ndcg={test_metrics['ndcg']:.4f}, test_coverage={test_metrics['coverage']:.4f}, pos_weight={baseline_mlp.pos_weight_:.2f}")

## 8. Sub-grupo `hyperparameter-tuning`

Sem grupo de feature-engineering (decisão explícita) -- só este sub-grupo,
irmão único dentro de `neural-models`.

In [ ]:
neural_tuning_group_run_id_path = MLFLOW_DIR / "neural_tuning_group_run_id.txt"

if neural_tuning_group_run_id_path.exists():
    NEURAL_TUNING_GROUP_ID = neural_tuning_group_run_id_path.read_text().strip()
    mlflow.start_run(run_id=NEURAL_TUNING_GROUP_ID, nested=True)
    print(f"Retomando grupo hyperparameter-tuning ({NEURAL_TUNING_GROUP_ID})")
else:
    neural_tuning_group_run = mlflow.start_run(run_name="hyperparameter-tuning", nested=True)
    NEURAL_TUNING_GROUP_ID = neural_tuning_group_run.info.run_id
    mlflow.set_tags({"category": NEURAL_CATEGORY_NAME})
    neural_tuning_group_run_id_path.write_text(NEURAL_TUNING_GROUP_ID)
    print(f"Grupo hyperparameter-tuning criado ({NEURAL_TUNING_GROUP_ID})")

# run_hyperparameter_search (seção 9) referência CATEGORY_NAME como global --
# mesma função usada em 01_eda.ipynb, sem alterações, só reaproveitada aqui.
CATEGORY_NAME = NEURAL_CATEGORY_NAME

## 9. Infra de busca (identica a `01_eda.ipynb`)

Mesmo design: cada trial é avaliado nos 3 splits por completo, sem uma run
"campeão" separada -- o melhor por `val_ndcg` só ganha a tag `is_best=true`.

In [ ]:
def tuning_exists(tuning_name):
    exp = mlflow.get_experiment_by_name(EXPERIMENT_NAME)
    if exp is None:
        return False
    existing = mlflow.search_runs(
        experiment_ids=[exp.experiment_id],
        filter_string=f"tags.mlflow.runName = 'tuning-{tuning_name}'",
    )
    return len(existing) > 0


def run_hyperparameter_search(tuning_name, model_class, search_space, n_trials,
                               train_feat_local, val_feat_local, test_feat_local, train_eval_feat_local,
                               feature_columns_local, fixed_params=None, seed=42):
    if tuning_exists(tuning_name):
        print(f"tuning-{tuning_name} já logado, pulando.")
        return None

    fixed_params = fixed_params or {}
    param_list = list(ParameterSampler(search_space, n_iter=n_trials, random_state=seed))
    X_tr = train_feat_local[feature_columns_local]
    y_tr = train_feat_local["label"]

    trial_run_ids = []
    trial_val_ndcgs = []

    with mlflow.start_run(run_name=f"tuning-{tuning_name}", nested=True) as tuning_parent:
        mlflow.set_tags({"category": CATEGORY_NAME, "tuning_target": tuning_name})

        for i, params in enumerate(param_list):
            full_params = {**fixed_params, **params}
            model = model_class(**full_params)
            model.fit(X_tr, y_tr)

            train_metrics = evaluate_model(model, train_eval_feat_local, feature_columns_local, all_items, k=10)
            val_metrics = evaluate_model(model, val_feat_local, feature_columns_local, all_items, k=10)
            test_metrics = evaluate_model(model, test_feat_local, feature_columns_local, all_items, k=10)

            with mlflow.start_run(run_name=f"trial-{i:03d}", nested=True) as trial_run:
                mlflow.set_tags({"category": CATEGORY_NAME, "tuning_target": tuning_name})
                mlflow.log_params(full_params)
                mlflow.log_metrics({f"train_{m}": v for m, v in train_metrics.items()})
                mlflow.log_metrics({f"val_{m}": v for m, v in val_metrics.items()})
                mlflow.log_metrics({f"test_{m}": v for m, v in test_metrics.items()})

            trial_run_ids.append(trial_run.info.run_id)
            trial_val_ndcgs.append(val_metrics["ndcg"])
            print(f"  [{tuning_name}] trial {i:02d}/{n_trials}: val_ndcg={val_metrics['ndcg']:.4f}, test_ndcg={test_metrics['ndcg']:.4f}")

        best_i = int(pd.Series(trial_val_ndcgs).idxmax())
        client = mlflow.MlflowClient()
        client.set_tag(trial_run_ids[best_i], "is_best", "true")
        print(f"[{tuning_name}] melhor trial (por val_ndcg): trial-{best_i:03d}")

        fig, ax = plt.subplots(figsize=(8, 4))
        ax.plot(range(len(trial_val_ndcgs)), trial_val_ndcgs, marker="o")
        ax.axhline(max(trial_val_ndcgs), color="gray", linestyle="--", linewidth=1)
        ax.scatter([best_i], [trial_val_ndcgs[best_i]], color="red", zorder=5, label="melhor (val)")
        ax.set_xlabel("trial")
        ax.set_ylabel("val NDCG@10")
        ax.set_title(f"Busca aleatória -- {tuning_name}")
        ax.legend()
        plt.tight_layout()
        mlflow.log_figure(fig, "search_convergence.png")
        plt.close(fig)

    client = mlflow.MlflowClient()
    best_run = client.get_run(trial_run_ids[best_i])
    return {
        "best_trial_run_id": trial_run_ids[best_i],
        "best_val_ndcg": trial_val_ndcgs[best_i],
        "test_metrics": {k: v for k, v in best_run.data.metrics.items() if k.startswith("test_")},
    }


print("Infra de tuning definida (reaproveitada de 01_eda.ipynb).")

## 10. Duas buscas -- `baseline` vs `tuned`

Mesmo espaço de busca (arquitetura, dropout, learning rate, weight decay,
batch size) e mesma seed no `ParameterSampler`, rodado duas vezes com
configurações de treino diferentes -- pra isolar o efeito de pos_weight +
patience maior do resto da busca (os 12 hiperparâmetros sorteados são
IDENTICOS nas duas):

- **`mlp-fe_v4-baseline`** -- configuração original: BCE sem peso de classe,
  `patience=5`.
- **`mlp-fe_v4-tuned`** -- com as duas melhorias: `pos_weight` automático
  (compensa o desbalanceamento 1:4 do negative sampling) e `patience=10`
  (menos propenso a parar cedo demais).

Ambas ficam lado a lado dentro de `hyperparameter-tuning`, cada uma com seus
12 trials -- o leaderboard pega o melhor entre as duas automaticamente.

In [ ]:
MLP_SEARCH_SPACE = {
    "hidden_dims": [(32,), (64,), (128,), (64, 32), (128, 64), (128, 64, 32)],
    "dropout": [0.0, 0.1, 0.2, 0.3, 0.4],
    "lr": [1e-4, 3e-4, 1e-3, 3e-3, 1e-2],
    "weight_decay": [0.0, 1e-5, 1e-4, 1e-3],
    "batch_size": [128, 256, 512, 1024],
}

print("Espaço de busca definido.")

In [ ]:
baseline_tuning_result = run_hyperparameter_search(
    tuning_name="mlp-fe_v4-baseline",
    model_class=NeuralMLPClassifier,
    search_space=MLP_SEARCH_SPACE,
    n_trials=12,
    train_feat_local=train_feat_v4, val_feat_local=val_feat_v4,
    test_feat_local=test_feat_v4, train_eval_feat_local=train_eval_feat_v4,
    feature_columns_local=FEATURE_COLUMNS_V4,
    fixed_params={"max_epochs": 100, "patience": 5, "weighted_loss": False, "seed": 42},
)

In [ ]:
tuned_tuning_result = run_hyperparameter_search(
    tuning_name="mlp-fe_v4-tuned",
    model_class=NeuralMLPClassifier,
    search_space=MLP_SEARCH_SPACE,
    n_trials=12,
    train_feat_local=train_feat_v4, val_feat_local=val_feat_v4,
    test_feat_local=test_feat_v4, train_eval_feat_local=train_eval_feat_v4,
    feature_columns_local=FEATURE_COLUMNS_V4,
    fixed_params={"max_epochs": 100, "patience": 10, "weighted_loss": True, "seed": 42},
)

In [ ]:
rows = []
for name, result in [("mlp-fe_v4-baseline", baseline_tuning_result), ("mlp-fe_v4-tuned", tuned_tuning_result)]:
    if result is None:
        continue
    row = {"search": name}
    row.update({k: v for k, v in result["test_metrics"].items()})
    rows.append(row)

pd.DataFrame(rows).set_index("search") if rows else print("Nada novo rodou nesta execução.")

## 11. Leaderboard -- mesmo mecanismo de `01_eda.ipynb`, apontando pra
`neural-models`

In [ ]:
def refresh_neural_leaderboard():
    exp = mlflow.get_experiment_by_name(EXPERIMENT_NAME)
    all_runs = mlflow.search_runs(experiment_ids=[exp.experiment_id])
    leaf_runs = all_runs[all_runs["metrics.test_ndcg"].notna()]
    client = mlflow.MlflowClient()

    def log_best_onto(run_id, subset, label):
        if subset.empty:
            print(f"[{label}] nenhum resultado ainda.")
            return
        best = subset.loc[subset["metrics.test_ndcg"].idxmax()]
        for metric in ["test_ndcg", "test_hit_rate", "test_mrr", "test_coverage"]:
            col = f"metrics.{metric}"
            if col in best and pd.notna(best[col]):
                client.log_metric(run_id, f"best_{metric}", best[col])
        client.set_tag(run_id, "best_run_name", best["tags.mlflow.runName"])
        if pd.notna(best.get("tags.tuning_target")):
            client.set_tag(run_id, "best_tuning_target", best["tags.tuning_target"])
        print(f"[{label}] melhor: {best['tags.mlflow.runName']} (test_ndcg={best['metrics.test_ndcg']:.4f})")

    category_leaf = leaf_runs[leaf_runs["tags.category"] == NEURAL_CATEGORY_NAME]
    log_best_onto(NEURAL_CATEGORY_RUN_ID, category_leaf, "neural-models (geral)")

    tuning_leaf = category_leaf[
        category_leaf["tags.tuning_target"].notna() & (category_leaf["tags.is_best"] == "true")
    ]
    log_best_onto(NEURAL_TUNING_GROUP_ID, tuning_leaf, "hyperparameter-tuning (neural)")


refresh_neural_leaderboard()

## 12. Fechar a categoria

In [ ]:
mlflow.end_run()  # encerra hyperparameter-tuning (restaura neural-models como ativa)
mlflow.end_run()  # encerra neural-models
print("Categoria neural-models fechada.")